In [2]:
import numpy as np 
import pandas as pd 
from sklearn.datasets import load_diabetes, make_regression 
from sklearn.feature_selection import SequentialFeatureSelector 
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_squared_error, r2_score 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler


In [5]:
# ===================================================================== # 
# 1\. CARREGAMENTO DOS DADOS (Dataset Diabetes com 10 preditores) 
# # ===================================================================== #

diabetes = load_diabetes(as_frame=True) 
X, y = diabetes.data, diabetes.target 

print(f"Chaves do dataset: {list(diabetes.keys())}")
print(f"Formato do dataset: {X.shape}")
print(f"Primeiras linhas do dataset:\n{X.head()}")
print(f"Valores alvo (target):\n{y.head()}")

# Divisão Treino / Teste (80% treino, 20% teste) 
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 ) 



Chaves do dataset: ['data', 'target', 'frame', 'DESCR', 'feature_names', 'data_filename', 'target_filename', 'data_module']
Formato do dataset: (442, 10)
Primeiras linhas do dataset:
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  
Valores alvo (target):
0    151.0
1     75.0
2    141.0
3    206.0
4    135.0
Name: target, dtype: float64


In [6]:
# Padronização dos preditores (prática recomendada) 
scaler = StandardScaler() 
X_train_scaled = pd.DataFrame( scaler.fit_transform(X_train), columns=X.columns, index=X_train.index ) 
X_test_scaled = pd.DataFrame( scaler.transform(X_test), columns=X.columns, index=X_test.index ) 

print(f"Primeiras linhas do dataset padronizado:\n{X_train_scaled.head()}")
print(f"Primeiras linhas do dataset de teste padronizado:\n{X_test_scaled.head()}")

Primeiras linhas do dataset padronizado:
          age       sex       bmi        bp        s1        s2        s3  \
17   1.498365  1.061370  0.219902  1.138874  0.728473  1.055893 -0.824451   
66  -0.228858  1.061370 -0.419366 -0.710591 -0.424929  0.272425 -1.529791   
137  0.085182 -0.942179  1.018987  1.992473 -0.309589 -0.326699 -0.119111   
245 -0.621409 -0.942179 -0.784662 -0.639458 -1.174640 -1.215508  0.664600   
31  -0.542899 -0.942179 -1.423930 -1.706457 -0.799784 -1.110167  1.291569   

           s4        s5        s6  
17   0.711038  0.547482 -0.061449  
66   1.484286 -0.019757  0.367236  
137 -0.062210  0.331237 -0.318660  
245 -0.835458 -1.069682 -2.719299  
31  -1.608706 -0.802859 -0.918820  
Primeiras linhas do dataset de teste padronizado:
          age       sex       bmi        bp        s1        s2        s3  \
287  0.948794 -0.942179 -0.168225 -0.354925  2.631586  2.649166  0.429487   
211  1.969426 -0.942179  0.745015  0.427541 -0.511434 -0.333282  0.037631   

In [7]:
# ===================================================================== 
# 2\. MODELO COMPLETO (Todas as 10 variáveis) 
# ===================================================================== 
lr_full = LinearRegression() 
lr_full.fit(X_train_scaled, y_train)

# Predição e avaliação do modelo completo
y_pred_full = lr_full.predict(X_test_scaled) 

# Métricas de avaliação do modelo completo
r2_full = r2_score(y_test, y_pred_full) 
rmse_full = np.sqrt(mean_squared_error(y_test, y_pred_full)) 

print(f"R² do modelo completo: {r2_full:.4f}")
print(f"RMSE do modelo completo: {rmse_full:.4f}")


R² do modelo completo: 0.4526
RMSE do modelo completo: 53.8534


In [8]:
# ===================================================================== 
# 3\. SELEÇÃO REGRESSIVA (SequentialFeatureSelector - Backward) 
# ===================================================================== 
sfs = SequentialFeatureSelector( 
    estimator=LinearRegression(), 
    n_features_to_select='auto', # Seleciona o número ideal via Validação Cruzada (5-fold) 
    direction='backward', # Começa com todas e remove uma a uma 
    scoring='neg_mean_squared_error', 
    cv=5, 
) 

# Treinamento do modelo de seleção regressiva
sfs.fit(X_train_scaled, y_train) 

# Variáveis mantidas e removidas 
selected_cols = list(X.columns[sfs.get_support()]) 
removed_cols = list(X.columns[~sfs.get_support()]) 

print(f"Variáveis mantidas: {selected_cols}")
print(f"Variáveis removidas: {removed_cols}")


Variáveis mantidas: ['sex', 'bmi', 'bp', 's1', 's5']
Variáveis removidas: ['age', 's2', 's3', 's4', 's6']


In [11]:
# ===================================================================== 
# 4\. MODELO REDUZIDO (Apenas as variáveis selecionadas pelo SFS) 
# ===================================================================== 
X_train_sfs = X_train_scaled[selected_cols] 
X_test_sfs = X_test_scaled[selected_cols] 

lr_sfs = LinearRegression() 
lr_sfs.fit(X_train_sfs, y_train) 

# Predição e avaliação do modelo reduzido
y_pred_sfs = lr_sfs.predict(X_test_sfs) 

r2_sfs = r2_score(y_test, y_pred_sfs) 
rmse_sfs = np.sqrt(mean_squared_error(y_test, y_pred_sfs)) 

print(f"R² do modelo reduzido: {r2_sfs:.4f}")
print(f"RMSE do modelo reduzido: {rmse_sfs:.4f}")

R² do modelo reduzido: 0.4611
RMSE do modelo reduzido: 53.4336


In [12]:
# ===================================================================== 
# 5\. IMPRESSÃO DOS RESULTADOS 
# ===================================================================== 
print('=========================================================') 
print(f'Total de Variáveis Originais ({len(X.columns)}): {list(X.columns)}') 
print( f'Variáveis Mantidas pelo SFS ({len(selected_cols)}): {selected_cols}' ) 
print(f'Variáveis Descartadas ({len(removed_cols)}): {removed_cols}') 
print('=========================================================\\n') 
print('--- RESULTADOS NO CONJUNTO DE TESTE ---') 
print(f'Modelo COMPLETO ({len(X.columns)} vars) -&gt; R²: {r2_full:.4f} | RMSE: {rmse_full:.2f}') 
print(f'Modelo FILTRADO ({len(selected_cols)} vars) -&gt; R²: {r2_sfs:.4f} | RMSE: {rmse_sfs:.2f}')

Total de Variáveis Originais (10): ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Variáveis Mantidas pelo SFS (5): ['sex', 'bmi', 'bp', 's1', 's5']
Variáveis Descartadas (5): ['age', 's2', 's3', 's4', 's6']
=========================================================\n
--- RESULTADOS NO CONJUNTO DE TESTE ---
Modelo COMPLETO (10 vars) -&gt; R²: 0.4526 | RMSE: 53.85
Modelo FILTRADO (5 vars) -&gt; R²: 0.4611 | RMSE: 53.43
